<a href="https://colab.research.google.com/github/szymonbloch/tuberculosis_detection/blob/main/preparing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import glob

COL_LABEL = 0
COL_DECISION = 2
COL_EXCLUDE = 5

LABELS_POS_NEG_PATH = '/content/drive/MyDrive/NTWI_project/data/Etykiety_1-layer-slides.xlsx'

df = pd.read_excel(LABELS_POS_NEG_PATH)
df.iloc[:, COL_DECISION] = df.iloc[:, COL_DECISION].astype(str).str.strip().str.lower()
df_one_layer_label = df[(df.iloc[:, COL_DECISION].isin(['pos', 'neg'])) &
                (df.iloc[:, COL_EXCLUDE].isnull())]

df_final = df_one_layer_label.iloc[:, [COL_LABEL, COL_DECISION]].copy()
df_final.columns = ['image_name', 'output_image']
df_final['output_image'] = df_final['output_image'].map({'neg': 0, 'pos': 1})
df_final['image_name'] = df_final['image_name'].str.split('.').str[0]


In [2]:
print(df_final.head())
len(df_final)

  image_name  output_image
0    AFB-001             1
1    AFB-002             1
2    AFB-003             1
3    AFB-004             1
5    AFB-006             0


136

In [3]:
ANOMALY_DIRECTORY_PATH = '/content/drive/MyDrive/NTWI_project/data/anomaly_detector/1-layer_ad_cnn_04_06_2025_00_31_21/*.csv'

csv_files = glob.glob(ANOMALY_DIRECTORY_PATH)
table_list = []

for file in csv_files:
  df_temp = pd.read_csv(file)
  df_temp.rename(columns={'image_name': 'tile_name', 'output': 'tile_output'}, inplace=True)

  df_temp['probability'] = df_temp['probability'].astype(str).str.extract(r'tensor\(([0-9.]+)').astype(float)
  df_temp['tile_name'] = df_temp['tile_name'].str.split('.').str[0].str.strip()
  df_temp['image_name'] = df_temp['tile_name'].str.split('_').str[0].str.strip()

  table_list.append(df_temp)

df_anomaly = pd.concat(table_list, ignore_index=True)


In [4]:
len(table_list)
print(df_anomaly.head())
len(df_anomaly)

               tile_name  tile_output  probability image_name
0  AFB-001_0_56832_33280            0       0.0007    AFB-001
1   AFB-001_0_5632_46592            0       0.0008    AFB-001
2   AFB-001_0_8960_36864            0       0.0008    AFB-001
3  AFB-001_0_41472_69888            0       0.0008    AFB-001
4  AFB-001_0_37888_47104            0       0.0007    AFB-001


4528219

In [5]:
COLORS_DIRECTORY_PATH = '/content/drive/MyDrive/NTWI_project/data/color_features/1_layer/*.csv'

csv_files = glob.glob(COLORS_DIRECTORY_PATH)
table_list = []

for file in csv_files:
  df_temp = pd.read_csv(file, header=None, sep=None, engine='python')
  df_temp.rename(columns={0: 'tile_name'}, inplace=True)
  df_temp['tile_name'] = df_temp['tile_name'].str.split('.').str[0].str.strip()

  table_list.append(df_temp)

df_colors = pd.concat(table_list, ignore_index=True)

In [6]:
len(df_colors)
print(df_colors.head())

                tile_name   1    2         3      4         5      6  \
0  AFB-136_0_10112_114432  82  252  14523061  65536  14850938  65536   
1  AFB-136_0_10112_115456  58  254  13782286  65536  14423841  65536   
2   AFB-136_0_10112_78592  54  253  14465245  65536  14948481  65536   
3   AFB-136_0_10112_78848  41  251  13409549  65536  14256694  65536   
4   AFB-136_0_10112_79104  47  250  12911066  65536  13974468  65536   

          7      8     9  ...  50  51  52  53  54  55  56  57  58  59  
0  15199481  65536  5519  ...   0   0   0   0   0   0   0   0   0 NaN  
1  15053318  65536  3753  ...   0   0   0   0   0   0   0   0   0 NaN  
2  15313635  65536  1691  ...   0   0   0   0   0   0   0   0   0 NaN  
3  14945436  65536   488  ...   0   0   0   0   0   0   0   0   0 NaN  
4  14912769  65536   231  ...   0   0   0   0   0   0   0   0   0 NaN  

[5 rows x 60 columns]


In [7]:
df_merged = df_anomaly.merge(df_final, on='image_name', how='inner').merge(df_colors, on='tile_name', how='inner')
df_merged = df_merged.dropna(axis=1, how='all') # Usunięcie ostatniej pustej kolumny

In [8]:
print("Liczba wierszy (kafli):", len(df_merged))
print("Liczba kolumn (cech):", len(df_merged.columns))
print(df_merged.head())

Liczba wierszy (kafli): 3155535
Liczba kolumn (cech): 63
               tile_name  tile_output  probability image_name  output_image  \
0  AFB-001_0_56832_33280            0       0.0007    AFB-001             1   
1   AFB-001_0_5632_46592            0       0.0008    AFB-001             1   
2   AFB-001_0_8960_36864            0       0.0008    AFB-001             1   
3  AFB-001_0_41472_69888            0       0.0008    AFB-001             1   
4  AFB-001_0_37888_47104            0       0.0007    AFB-001             1   

     1    2         3      4         5  ...  49  50  51  52  53  54  55  56  \
0   93  255  15076976  65536  14964763  ...   0   0   0   0   0   0   0   0   
1   93  255  15059846  65536  14919858  ...   0   0   0   0   0   0   0   0   
2   77  255  15069769  65536  14935905  ...   0   0   0   0   0   0   0   0   
3  131  255  15171065  65536  14931638  ...   0   0   0   0   0   0   0   0   
4   52  255  14992676  65536  14798151  ...   0   0   0   0   0   0   0  

In [12]:
statistic_dict = {
    'probability': ['mean', 'max', 'std'],
    'output_image': ['first']
}

for col in df_merged.columns:
    if type(col) == int or str(col).isdigit():
        statistic_dict[col] = ['mean']
df_slide_level = df_merged.groupby('image_name').agg(statistic_dict).reset_index()
df_slide_level.columns = ['_'.join([str(c) for c in col]).strip('_') for col in df_slide_level.columns.values]
df_slide_level.rename(columns={'output_image_first': 'output_image'}, inplace=True)

In [18]:
print("Liczba slajdów (pacjentów):", len(df_slide_level))
print("Liczba statystycznych cech:", len(df_slide_level.columns))
print(df_slide_level.head())

Liczba slajdów (pacjentów): 133
Liczba statystycznych cech: 63
  image_name  probability_mean  probability_max  probability_std  \
0    AFB-001          0.000828           0.0046         0.000193   
1    AFB-002          0.001145           0.0038         0.000278   
2    AFB-003          0.000870           0.0018         0.000172   
3    AFB-004          0.001611           0.0039         0.000566   
4    AFB-006          0.000744           0.0016         0.000231   

   output_image      1_mean      2_mean        3_mean        4_mean  \
0             1   96.281867  254.673199  1.467378e+07  64700.453633   
1             1   81.000502  254.821320  1.401996e+07  65389.330018   
2             1  102.205684  254.836891  1.461163e+07  64883.599532   
3             1   37.353746  248.508900  1.138322e+07  65088.762927   
4             0   50.412028  245.569416  1.109618e+07  64604.426957   

         5_mean  ...  49_mean  50_mean  51_mean  52_mean  53_mean  54_mean  \
0  1.447040e+07  ...   

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# 2. Podział na cechy (X) i ostateczną diagnozę (y)
X = df_slide_level.drop(columns=['image_name', 'output_image'])
y = df_slide_level['output_image']

# 3. Odłożenie 20% pacjentów do testowania modelu
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Budowa i trening Głównego Lekarza (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 5. Egzamin na ukrytych danych
y_pred = rf_model.predict(X_test)
bacc = balanced_accuracy_score(y_test, y_pred)


In [17]:
print(f"\nWynik Balanced Accuracy: {bacc * 100:.2f}%")
print("Macierz pomyłek (Prawdziwe vs Przewidziane):\n", confusion_matrix(y_test, y_pred))


Wynik Balanced Accuracy: 79.12%
Macierz pomyłek (Prawdziwe vs Przewidziane):
 [[15  2]
 [ 3  7]]
